# 离散程度的度量指标完整教程：从全距到标准差

## 📚 教学目标
1. 理解全距、方差、标准差的定义和计算公式
2. 掌握总体方差 ($\sigma^2$) vs 样本方差 ($s^2$) 的区别（分母 n vs n-1）
3. 手算 10 个数据点的方差和标准差
4. 用 scipy 验证，扩展到 500 个数据点
5. 理解范围经验法则和切比雪夫定理的应用

**参考书**：《基础统计学(第14版)》（Triola）第 3-2 节
**教学策略**：先用极小数据集让你「看见」每一步计算，再扩展到真实规模

---

## 1. 为什么需要度量离散程度？

### 🎯 一个直觉问题

两组学生的考试成绩均值都是 75 分：
- **A 组**：74, 75, 76, 74, 76, 75, 75, 74, 76, 75
- **B 组**：50, 95, 60, 85, 70, 80, 55, 90, 65, 100

均值相同，但两组的数据分布**完全不同**！A 组成绩集中，B 组成绩分散。

### 📖 书中的定义

> 离散程度是统计学中最重要的一个主题。离散程度的度量指标用于衡量数据值偏离其均值的程度。
> 通过降低商品和/或服务的离散程度来提高其质量，是企业的重要目标。

接下来我们将学习三种核心离散程度度量指标：**全距**、**方差**、**标准差**。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

np.random.seed(42)
print('✅ 库导入完成')

---

## 2. 全距 (Range)：最简单的离散度量

### 📐 定义

$$\text{全距} = \text{最大值} - \text{最小值}$$

### 📖 书中的要点

> 全距的计算只涉及最大值和最小值，因此它对异常值非常灵敏，即不具备抗性。
> 由于全距只用到两个值，无法考虑数据中的每一个值，因此它不能真实地反映所有数据值之间的离散程度。

In [ ]:
wait_times = np.array([50, 25, 75, 35, 50, 25, 30, 50, 45, 25, 20])

print('📋 数据：飞越太空山等候时间（分钟）')
print(f'  原始数据: {wait_times}')
print(f'  排序后:   {np.sort(wait_times)}')
print(f'  样本量 n = {len(wait_times)}')

In [ ]:
data_max = wait_times.max()
data_min = wait_times.min()
range_val = data_max - data_min

print(f'📊 步骤 1: 计算全距')
print(f'  最大值 = {data_max}')
print(f'  最小值 = {data_min}')
print(f'  全距 = 最大值 - 最小值 = {data_max} - {data_min} = {range_val:.1f} 分钟')
print(f'\n💡 全距为 {range_val:.1f} 分钟，只反映了极端值的差距，不能反映中间数据的离散情况')

---

## 3. 样本方差与标准差：核心离散度量

### 📐 样本方差公式

$$s^2 = \frac{\sum(x_i - \bar{x})^2}{n - 1}$$

其中：
- $x_i$ = 每个数据值
- $\bar{x}$ = 样本均值
- $n$ = 样本量
- $n-1$ = 自由度（用 $n-1$ 而非 $n$，是为了得到总体方差的无偏估计）

### 📐 样本标准差公式

$$s = \sqrt{s^2} = \sqrt{\frac{\sum(x_i - \bar{x})^2}{n - 1}}$$

### 💡 标准差的重要性质

| 性质 | 说明 |
|------|------|
| 非负性 | $s \geq 0$，只有当所有数据值完全相同时 $s = 0$ |
| 离散度量 | $s$ 越大，数据越分散 |
| 异常值敏感 | 异常值会急剧增大标准差 |
| 单位一致 | 标准差的单位与原始数据相同（如分钟、厘米） |

### 📖 总体方差 vs 样本方差

| | 总体方差 $\sigma^2$ | 样本方差 $s^2$ |
|---|---|---|
| 分母 | $N$（总体大小） | $n-1$（自由度） |
| 适用场景 | 拥有全部数据 | 只有样本数据 |
| 符号 | $\sigma^2$, $\sigma$ | $s^2$, $s$ |
| 无偏性 | 精确值 | $s^2$ 是 $\sigma^2$ 的无偏估计 |

> **为什么分母是 $n-1$？** 因为用样本估计总体时，$n-1$ 能使样本方差成为总体方差的无偏估计。

In [ ]:
n = len(wait_times)

x_bar = wait_times.sum() / n
print(f'📊 步骤 1: 计算均值')
print(f'  Σx = {wait_times.sum()}')
print(f'  n = {n}')
print(f'  x̄ = Σx / n = {wait_times.sum()} / {n} = {x_bar:.4f}')

In [ ]:
deviations = wait_times - x_bar
print(f'📊 步骤 2: 计算偏差 (x_i - x̄)')
for i, (x, d) in enumerate(zip(wait_times, deviations)):
    print(f'  x_{i+1} - x̄ = {x} - {x_bar:.4f} = {d:>8.4f}')

In [ ]:
sq_deviations = deviations ** 2
print(f'📊 步骤 3: 计算偏差的平方 (x_i - x̄)²')
for i, (d, sd) in enumerate(zip(deviations, sq_deviations)):
    print(f'  ({d:>8.4f})² = {sd:>10.4f}')

In [ ]:
ss = sq_deviations.sum()
print(f'📊 步骤 4: 计算平方和 Σ(x_i - x̄)²')
print(f'  SS = {ss:.4f}')

In [ ]:
s_squared = ss / (n - 1)
print(f'📊 步骤 5: 计算样本方差 s²')
print(f'  s² = SS / (n-1) = {ss:.4f} / {n-1} = {s_squared:.4f}')

In [ ]:
s_manual = np.sqrt(s_squared)
print(f'📊 步骤 6: 计算样本标准差 s')
print(f'  s = √(s²) = √({s_squared:.4f}) = {s_manual:.4f}')
print(f'\n🎯 结果: s = {s_manual:.1f} 分钟（比原始数据多 1 位小数）')

In [ ]:
s_scipy = np.std(wait_times, ddof=1)
var_scipy = np.var(wait_times, ddof=1)

print('🔬 scipy 验证:')
print(f'  手算方差   s² = {s_squared:.6f}')
print(f'  scipy 方差    = {var_scipy:.6f}')
print(f'  手算标准差 s  = {s_manual:.6f}')
print(f'  scipy 标准差  = {s_scipy:.6f}')
print(f'\n  ✅ 验证通过！')

---

## 4. 可视化：均值 ± 标准差

In [ ]:
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.scatter(wait_times, np.ones(n), s=80, color='steelblue', edgecolors='black', zorder=5)
ax1.axvline(x_bar, color='#e74c3c', linewidth=2, linestyle='-', label=f'Mean = {x_bar:.1f}')
ax1.axvspan(x_bar - s_manual, x_bar + s_manual, alpha=0.15, color='#2ecc71',
            label=f'±1s = [{x_bar-s_manual:.1f}, {x_bar+s_manual:.1f}]')
ax1.axvline(x_bar - s_manual, color='#2ecc71', linewidth=1.5, linestyle='--', alpha=0.7)
ax1.axvline(x_bar + s_manual, color='#2ecc71', linewidth=1.5, linestyle='--', alpha=0.7)
ax1.set_xlabel('Wait Time (min)', fontsize=12)
ax1.set_yticks([])
ax1.set_title('Dot Plot with Mean ± 1 SD', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='x', alpha=0.3)

group_a = np.array([74, 75, 76, 74, 76, 75, 75, 74, 76, 75])
group_b = np.array([50, 95, 60, 85, 70, 80, 55, 90, 65, 100])

ax2 = axes[1]
x_pos = [0, 1]
means = [group_a.mean(), group_b.mean()]
stds = [group_a.std(ddof=1), group_b.std(ddof=1)]

bars = ax2.bar(x_pos, means, yerr=stds, capsize=10, color=['#3498db', '#e74c3c'],
               alpha=0.8, edgecolor='black', error_kw={'linewidth': 2})
ax2.set_xticks(x_pos)
ax2.set_xticklabels([f'Group A\n(std={stds[0]:.1f})', f'Group B\n(std={stds[1]:.1f})'], fontsize=11)
ax2.set_ylabel('Score', fontsize=12)
ax2.set_title('Same Mean, Different Dispersion', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/3.2_fig1.png', dpi=100, bbox_inches='tight')
plt.close()
print('✅ 图表已保存')

print(f'\n💡 左图：绿色区域为均值 ± 1 个标准差的范围')
print(f'   右图：两组均值都是 75 分，但 A 组标准差仅 {stds[0]:.1f}，B 组高达 {stds[1]:.1f}')

---

## 5. 扩展到 500 个数据点

In [ ]:
n_large = 500
wait_large = np.random.normal(loc=40, scale=15, size=n_large)
wait_large = np.clip(wait_large, 5, 120)

mean_large = wait_large.mean()
std_large = wait_large.std(ddof=1)
var_large = wait_large.var(ddof=1)
range_large = wait_large.max() - wait_large.min()

print('=' * 55)
print('📋 500 个等候时间数据的离散程度统计')
print('=' * 55)
print(f'\n📊 基本统计量:')
print(f'  样本量 n     = {n_large}')
print(f'  均值 x̄       = {mean_large:.2f} 分钟')
print(f'  最小值       = {wait_large.min():.2f} 分钟')
print(f'  最大值       = {wait_large.max():.2f} 分钟')
print(f'\n📐 离散程度指标:')
print(f'  全距         = {range_large:.2f} 分钟')
print(f'  样本方差 s²  = {var_large:.2f}')
print(f'  样本标准差 s = {std_large:.2f} 分钟')

In [ ]:
within_1s = np.sum((wait_large >= mean_large - std_large) & (wait_large <= mean_large + std_large)) / n_large
within_2s = np.sum((wait_large >= mean_large - 2*std_large) & (wait_large <= mean_large + 2*std_large)) / n_large
within_3s = np.sum((wait_large >= mean_large - 3*std_large) & (wait_large <= mean_large + 3*std_large)) / n_large

print('📊 范围经验法则验证:')
print(f'  ±1s [{mean_large-std_large:.1f}, {mean_large+std_large:.1f}]: {within_1s*100:.1f}% (理论 ≈ 68.27%)')
print(f'  ±2s [{mean_large-2*std_large:.1f}, {mean_large+2*std_large:.1f}]: {within_2s*100:.1f}% (理论 ≈ 95.45%)')
print(f'  ±3s [{mean_large-3*std_large:.1f}, {mean_large+3*std_large:.1f}]: {within_3s*100:.1f}% (理论 ≈ 99.73%)')

---

## 6. 范围经验法则与切比雪夫定理

### 📐 范围经验法则 (Empirical Rule)

适用于**近似正态**分布的数据：

| 范围 | 数据占比 |
|------|----------|
| $\bar{x} \pm 1s$ | 约 68% |
| $\bar{x} \pm 2s$ | 约 95% |
| $\bar{x} \pm 3s$ | 约 99.7% |

### 📐 切比雪夫定理 (Chebyshev's Theorem)

适用于**任何**分布的数据（更保守）：

$$\text{至少} \quad 1 - \frac{1}{k^2} \quad \text{的数据落在} \quad \bar{x} \pm ks \quad \text{内}$$

In [ ]:
print('📊 范围经验法则 vs 切比雪夫定理')
print('=' * 65)
print(f'{"k":>4} | {"观测占比":>10} | {"经验法则":>10} | {"切比雪夫":>12} | {"适用"}')
print('-' * 65)

for k in [1, 1.5, 2, 2.5, 3]:
    observed = np.sum((wait_large >= mean_large - k*std_large) & 
                      (wait_large <= mean_large + k*std_large)) / n_large * 100
    empirical = {1: 68.27, 2: 95.45, 3: 99.73}.get(k, float('nan'))
    chebyshev = (1 - 1/k**2) * 100 if k > 1 else float('nan')
    
    emp_str = f'{empirical:.1f}%' if not np.isnan(empirical) else 'N/A'
    cheb_str = f'{chebyshev:.1f}%' if not np.isnan(chebyshev) else 'N/A'
    
    print(f'  {k:.1f} | {observed:>8.1f}% | {emp_str:>10} | {cheb_str:>12} | {"正态" if k in [1,2,3] else "任意"}')

print('\n💡 观测值接近经验法则（正态假设下），且始终满足切比雪夫下界')

---

## 7. 异常值对标准差的影响

In [ ]:
original = np.array([50, 25, 75, 35, 50, 25, 30, 50, 45, 25, 20])
with_outlier = np.append(original, 200)

print('📊 异常值对标准差的影响')
print('=' * 50)
print(f'  原始数据 (n=11): {original}')
print(f'  加入异常值 (n=12): ...200')
print()
print(f'  {"指标":<15} {"原始数据":>12} {"加入异常值":>12} {"变化":>10}')
print(f'  {"-"*49}')
print(f'  {"均值":<15} {original.mean():>12.2f} {with_outlier.mean():>12.2f} {with_outlier.mean()-original.mean():>+10.2f}')
print(f'  {"标准差":<15} {original.std(ddof=1):>12.2f} {with_outlier.std(ddof=1):>12.2f} {with_outlier.std(ddof=1)-original.std(ddof=1):>+10.2f}')
print(f'  {"全距":<15} {original.max()-original.min():>12.2f} {with_outlier.max()-with_outlier.min():>12.2f} {(with_outlier.max()-with_outlier.min())-(original.max()-original.min()):>+10.2f}')
print(f'\n💡 仅一个异常值（200 分钟），标准差从 {original.std(ddof=1):.1f} 增加到 {with_outlier.std(ddof=1):.1f}')

---

## 📌 核心概念回顾

### 📌 全距 (Range)
- **定义**: 最大值与最小值之差
- **公式**: $\text{Range} = x_{\max} - x_{\min}$
- **缺点**: 对异常值极度敏感，不具备抗性

### 📌 样本方差 (Sample Variance)
- **公式**: $s^2 = \frac{\sum(x_i - \bar{x})^2}{n - 1}$
- **含义**: 偏差平方的平均（自由度 $n-1$）

### 📌 样本标准差 (Sample Standard Deviation)
- **公式**: $s = \sqrt{s^2}$
- **含义**: 衡量数据偏离均值的平均距离

### 🔗 完整流程
```
原始数据 → 计算均值 → 求偏差 → 平方偏差 → 求和 → 除以(n-1) → 开方 → 标准差
```

---

## ❌ 常见误区

### ❌ 误区 1: 标准差和方差的分母都是 n
**✓ 正确理解**: 样本方差/标准差的分母是 $n-1$（自由度），总体方差/标准差的分母才是 $N$。

### ❌ 误区 2: 标准差可以为负
**✓ 正确理解**: 标准差 $s \geq 0$。只有当所有数据值完全相同时 $s = 0$。

### ❌ 误区 3: 均值相同，数据分布也相同
**✓ 正确理解**: 均值只反映集中趋势，不反映离散程度。必须同时看均值和标准差。

### ❌ 误区 4: 标准差大一定不好
**✓ 正确理解**: 标准差的「好坏」取决于具体场景。投资中高波动意味着高风险；产品质量中高标准差意味着不稳定。

### ❌ 误区 5: 经验法则适用于所有数据
**✓ 正确理解**: 范围经验法则（68-95-99.7）只适用于近似正态分布的数据。对于偏态数据应使用切比雪夫定理。